In [1]:
!git clone https://github.com/elenipapadopulos/NLP_LAB3_Datasets.git

Cloning into 'NLP_LAB3_Datasets'...
remote: Enumerating objects: 11, done.
remote: Counting objects: 100% (11/11), done.
remote: Compressing objects: 100% (9/9), done.
remote: Total 11 (delta 3), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (11/11), 6.23 KiB | 1.25 MiB/s, done.
Resolving deltas: 100% (3/3), done.


In [2]:
import nltk
nltk.download('punkt_tab')
nltk.download('brown') # import the Brown corpus from NLTK
nltk.download('punkt') # import tokenizer

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package brown to /root/nltk_data...
[nltk_data]   Unzipping corpora/brown.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [3]:
from nltk import ngrams

In [4]:
from nltk.corpus import brown

sentences = brown.sents()[:10000]
tokenized_corpus = []

for sentence in sentences:
  tokens = nltk.word_tokenize(' '.join(sentence))

  tokens = ['<s>'] + tokens + ['</s>']

  tokenized_corpus.extend(tokens)

In [5]:
from nltk import bigrams
from collections import defaultdict

bigram = list(bigrams(tokenized_corpus))

# build a bigram model (dictionary of counts values)
bigram_model = defaultdict(lambda: defaultdict(lambda: 0))

In [6]:
# count frequency of co-occurrence
for w1, w2 in bigram:
    bigram_model[w1][w2] += 1

w1 = ('there')
print(f"{w1} -> {dict(bigram_model[w1])}")

there -> {'should': 3, 'was': 35, 'were': 30, 'also': 1, 'to': 3, '``': 3, 'would': 9, 'will': 12, 'has': 9, 'are': 30, 'be': 4, 'is': 86, '.': 11, 'seemed': 2, 'existed': 2, "'s": 2, 'pitching': 1, 'before': 1, 'and': 3, 'surely': 1, 'for': 2, 'one': 1, 'while': 1, 'any': 1, 'had': 3, '</s>': 3, ',': 6, 'never': 2, 'that': 1, 'ever': 1, 'Monday': 1, 'could': 5, 'she': 1, 'anything': 1, 'do': 1, 'appeared': 1, 'must': 3, 'he': 3, 'appears': 1, 'as': 1, 'may': 1, 'represents': 1, 'shall': 1, 'it': 1, 'remains': 1, 'can': 3, 'in': 1, 'on': 1, 'remembering': 1, 'lies': 1, 'whether': 1, 'have': 2, 'of': 1, 'might': 1, 'still': 1, 'seems': 1, 'lay': 1, 'so': 1, 'been': 1, 'an': 1, 'tends': 1, 'considerably': 1, ';': 1, 'about': 1, 'pleading': 1, 'anymore': 1}


In [7]:
# transform counts into probabilities
for w1 in bigram_model:
    total_count = float(sum(bigram_model[w1].values()))
    for w2 in bigram_model[w1]:
        bigram_model[w1][w2] /= total_count

w1 = ('there')
print(f"{w1} -> {dict(bigram_model[w1])}")

# bigram_model[w1] is a probability distribuition over the token that follows w1

there -> {'should': 0.00949367088607595, 'was': 0.11075949367088607, 'were': 0.0949367088607595, 'also': 0.0031645569620253164, 'to': 0.00949367088607595, '``': 0.00949367088607595, 'would': 0.028481012658227847, 'will': 0.0379746835443038, 'has': 0.028481012658227847, 'are': 0.0949367088607595, 'be': 0.012658227848101266, 'is': 0.2721518987341772, '.': 0.03481012658227848, 'seemed': 0.006329113924050633, 'existed': 0.006329113924050633, "'s": 0.006329113924050633, 'pitching': 0.0031645569620253164, 'before': 0.0031645569620253164, 'and': 0.00949367088607595, 'surely': 0.0031645569620253164, 'for': 0.006329113924050633, 'one': 0.0031645569620253164, 'while': 0.0031645569620253164, 'any': 0.0031645569620253164, 'had': 0.00949367088607595, '</s>': 0.00949367088607595, ',': 0.0189873417721519, 'never': 0.006329113924050633, 'that': 0.0031645569620253164, 'ever': 0.0031645569620253164, 'Monday': 0.0031645569620253164, 'could': 0.015822784810126583, 'she': 0.0031645569620253164, 'anything':

In [8]:
!pip install transformers

In [9]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer
import transformers
import torch
import math
import numpy as np

from transformers import AutoTokenizer, AutoModelForCausalLM
import matplotlib.pyplot as plt
import pandas as pd

In [10]:
device =  "cuda:0" if torch.cuda.is_available() else "cpu"

model = "gpt2"

tokenizer = GPT2Tokenizer.from_pretrained(model)
model = GPT2LMHeadModel.from_pretrained(model, pad_token_id = tokenizer.eos_token_id).to(device)

tokenizer.pad_token = tokenizer.eos_token # set the pad to eos_token
model.config.pad_token_id = tokenizer.pad_token_id

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [11]:
#logits?

# exercise 1: build a trigram model with backoff

In [12]:
from collections import Counter, defaultdict

unigram = list(ngrams(tokenized_corpus, 1))

unigram_count = Counter(unigram) # computes the occurrences

**Ex.1.1**: Let's start by computing the unigram distribution: we already have the unigram counts, so what's left to do is normalizing them.

In [13]:
## compute the total number of tokens
total_number_of_tokens=sum(unigram_count.values())

## create a unigram model dictionary
## key: word
## value: probability

unigram_model={}
for word in unigram_count:
    unigram_model[word]=unigram_count[word]/total_number_of_tokens

# ------- here i have modified thissssss -------
w1 = ('there',)
print(f"{w1} -> {unigram_model[w1]}")

('there',) -> 0.0013045560381128524


In [14]:
for sentence in sentences:

  tokens = nltk.word_tokenize(' '.join(sentence))

  tokens = ['<s>', '<s>'] + tokens + ['</s>']

  tokenized_corpus.extend(tokens)

In [15]:
from nltk import trigrams

trigram = list(trigrams(tokenized_corpus))

# let's define the trigram model as a dictionary of dictionaries
trigram_model = defaultdict(lambda: defaultdict(lambda: 0))

**Ex 1.2**: Compute co-occurrences counts. Note that the dictionary trigram_model takes as input a tuple of words as you can see from the example in the cell.

In [16]:
## Ex 1.2

## compute trigram counts
trigram_count = Counter(trigram)
for w1, w2, w3 in trigram:
    trigram_model[(w1, w2)][w3] += 1

w1_w2 = ('there', 'were')
print(f"{w1_w2} -> {dict(trigram_model[w1_w2])}")

('there', 'were') -> {'no': 4, 'lots': 2, 'several': 2, 'such': 4, 'not': 4, 'only': 2, "n't": 2, 'plenty': 2, 'as': 2, ',': 2, 'other': 2, 'in': 4, 'those': 6, 'moderates': 2, 'to': 4, 'a': 2, 'actually': 2, 'one': 2, 'solid': 2, 'evidences': 2, 'flashes': 4, 'very': 2}


**Ex 1.3**: Compute trigram probabilities by normalizing counts.

In [17]:
## Ex 1.3

## compute trigram probabilities
for w1_w2 in trigram_model:
    total_count=float(sum(trigram_model[w1_w2].values()))
    if total_count>0:  # Add this check
        for w3 in trigram_model[w1_w2]:
            trigram_model[w1_w2][w3]=trigram_model[w1_w2][w3]/total_count
    else:
        for w3 in trigram_model[w1_w2]:
            trigram_model[w1_w2][w3]=0

w1_w2 = ('there', 'were')
print(f"{w1_w2} -> {dict(trigram_model[w1_w2])}")

('there', 'were') -> {'no': 0.06666666666666667, 'lots': 0.03333333333333333, 'several': 0.03333333333333333, 'such': 0.06666666666666667, 'not': 0.06666666666666667, 'only': 0.03333333333333333, "n't": 0.03333333333333333, 'plenty': 0.03333333333333333, 'as': 0.03333333333333333, ',': 0.03333333333333333, 'other': 0.03333333333333333, 'in': 0.06666666666666667, 'those': 0.1, 'moderates': 0.03333333333333333, 'to': 0.06666666666666667, 'a': 0.03333333333333333, 'actually': 0.03333333333333333, 'one': 0.03333333333333333, 'solid': 0.03333333333333333, 'evidences': 0.03333333333333333, 'flashes': 0.06666666666666667, 'very': 0.03333333333333333}


**Ex. 1.4** Implement the back-off function.

The function backoff should take as input the trigram, bigram and unigram models and the context words $w_1$ and $w_2$ and it should return the next word $w_3$ to be generated and its probabilty.


Recall: for each $w_3$, if $P(w_3 \mid w_1, w_2)$ is not in the trigram model, then use $P(w_3 \mid w_2)$. If there is no $P(w_3 \mid w_2)$, then use $P(w_3)$.

In [18]:
def backoff(trigram_model, bigram_model, unigram_model, w1, w2):

  # given the context w1_w2, use backoff to find word with the highest probability:
  # for every w in vocabulary, check if p(w|w1_w2) exists, otherwise back off to bigrams
  # if there is no p(w|w2), then back off to unigram probability

  # use the computed trigram, bigram and unigram models

  # greedy decoding: you should return next_word, probability

  w1_w2=(w1,w2)
  for w3 in trigram_model[w1_w2]:
    if trigram_model[w1_w2][w3] > 0:
      return w3, trigram_model[w1_w2][w3]
    else:
      if bigram_model[w2][w3] > 0:
        return w3, bigram_model[w2][w3]
      else:
        return w3, unigram_model[w3]

In [19]:
def backoff(trigram_model, bigram_model, unigram_model, w1, w2):
    w1_w2=(w1, w2)

    if w1_w2 in trigram_model and trigram_model[w1_w2]:
        w3=max(trigram_model[w1_w2], key=lambda w: trigram_model[w1_w2][w])
        return w3, trigram_model[w1_w2][w3]

    if w2 in bigram_model and bigram_model[w2]:
        w3=max(bigram_model[w2], key=lambda w: bigram_model[w2][w])
        return w3, bigram_model[w2][w3]

    if unigram_model:
        w3=max(unigram_model, key=lambda w: unigram_model[w])
        return w3, unigram_model[w3]

    return None, 0

In [20]:
def backoff_generation(trigram_model, bigram_model, unigram_model, seed_words, max_lenght=15):

  w1, w2 = seed_words
  generated_words = [w1, w2]

  log_prob_sum = 0.0
  N = 0

  for _ in range(max_lenght):

    next_word, probability = backoff(trigram_model, bigram_model, unigram_model, w1, w2)

    generated_words.append(next_word)
    log_prob_sum += math.log(probability)
    N += 1

    w1, w2 = w2, next_word

  perplexity = math.exp(-log_prob_sum / N) if N > 0 else float('inf')

  return ' '.join(generated_words), perplexity

In [21]:
seed = ("they", "care")
generated_sentence, perplexity = backoff_generation(trigram_model, bigram_model, unigram_model, seed)
print("Generated sentence:", generated_sentence)
print("Perplexity:", perplexity)

Generated sentence: they care for the first time in the first time in the first time in the first
Perplexity: 8.278479151107373


In [22]:
seed = ("there", "was")
generated_sentence, perplexity = backoff_generation(trigram_model, bigram_model, unigram_model, seed)
print("Generated sentence:", generated_sentence)
print("Perplexity:", perplexity)

Generated sentence: there was no problem . </s> <s> <s> The President said he was the first time in
Perplexity: 4.667583943291664


### Ex. 2: typo detection

In [23]:
df = pd.read_csv("/content/NLP_LAB3_Datasets/typo_dataset1.csv")

In [24]:
model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


We will use the token \<s> as our marker for the beginning of a sentence. First, we need to add it to the tokenizer’s vocabulary, then set it as the bos_token, and finally, resize the model’s embedding size.

In [25]:
bos_token = "<s>"
tokenizer.add_tokens([bos_token])
tokenizer.bos_token = bos_token
model.resize_token_embeddings(len(tokenizer))

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Embedding(50258, 768)

**Ex 3.1** Write a function that returns the log-probability assigned to each generated token.

You can use the `get_next_word_probs` function as a reference, but remember that this time we’re focusing on the distribution of tokens **within** the sentence, rather than the probability distribution of the next token. You can follow the comments we left in the box to guide you in the implementation.

In [26]:
def get_next_word_probs(model, input):
  input_ids = tokenizer.encode(input, return_tensors='pt').to(device)

  with torch.no_grad():
    logits = model(input_ids).logits.squeeze()[-1]
    # logits has shape [1, input_len, model_size]
    # we compress the size into [input_len, model_size] with squeeze()
    # we are interested in next token generation: we access only the last logit

  probabilities = torch.nn.functional.softmax(logits, dim=0) # compute probabilities using softmax
  return probabilities

  #the logit refers to the generation of following words

prefix = "My name is"
probabilities = get_next_word_probs(model, prefix)
top_token_probs, top_token_vals = torch.topk(probabilities, 10) # get the top 10 tokens with the highest probabilities

for token, prob in zip(top_token_vals, top_token_probs):
  print("%.3f" % prob.item(), tokenizer.decode(token))

  #it takes the vector seen before and computes the avg

0.011  John
0.008  James
0.008  David
0.007  Michael
0.006  K
0.005  L
0.005  the
0.005  a
0.005  J
0.005  Daniel


In [27]:
def get_token_logprobs(sentence):
    ## tokenize the sentence, compute the output and retrieve logits
    inputs=tokenizer(sentence, return_tensors="pt")

    outputs=model(**inputs)
    logits=outputs.logits

    ## remove the logits relative to the last token: we are not interested in next token generation
    ## expected shape: (sen_len, model_size) (suggestion: use squeeze))
    logits=logits.squeeze(0)[:-1, :]  # shape: (seq_len-1, vocab_size)

    # retrieve the indices (input_ids) of the sentence
    indices=inputs["input_ids"].squeeze(0)[1:]

    # compute lo-probabilites
    log_probs=torch.nn.functional.log_softmax(logits, dim=-1)

    ## retrieve the probabilities of the tokens
    token_logprobs=log_probs[range(len(indices)), indices]

    ## convert input ids to tokens to obtain a list of tokens
    tokens=tokenizer.convert_ids_to_tokens(inputs["input_ids"].squeeze(0))[1:]

    return tokens, token_logprobs.tolist()

Ex 3.2 Write a function that returns the cumulative log-probability up to each token in a sentence, using the probabilities computed before.

Remind that, as we are considering log-probabilities, you should sum the individual log-probabilities of each individual token.

Hint: you could return a list of elements like (w, cumulative_probability_up_to_w)

In [28]:
def get_cumulative_token_logprobs(sentence):
    tokens, token_logprobs=get_token_logprobs(sentence)

    cumulative_logprobs=[]
    current_sum=0

    for token, logprob in zip(tokens, token_logprobs):
        current_sum += logprob
        cumulative_logprobs.append((token, current_sum))

    return cumulative_logprobs

**Ex. 3.3** Write a function that determines whether a sentence contains typos or not based on the difference of log-probabilities of consecutive tokens/words. The hypothesis is that a significant drop in probability between consecutive tokens may indicate that the model did not expect that token, potentially signaling a typo.

To implement this, we can define a threshold: if the difference between the log-probabilities of consecutive tokens exceeds this threshold, we can assume the sentence likely contains a typo.

The function should take as input the sentence to be analyzed and the threshold to apply for detection and it should return 0 if the sentence is flagged as potentially containing typos or	1 if the sentence is considered correct.

In the function you should confront consecutive log-probabilities and check whether, for at least one pair, their difference is above the set threshold: in that case, classify the whole sentence as incorrect.

In [29]:
def detect_typos(sentence, threshold=5):
    cumulative_logprobs=get_cumulative_token_logprobs(sentence)

    logprobs=[logprob for _, logprob in cumulative_logprobs]

    #the sentence has only one token > it can't have consecutive differences
    if len(logprobs) <= 1:
        return 1

    differences=[]
    for i in range(1, len(logprobs)):
        diff=logprobs[i-1]-logprobs[i]
        differences.append(diff)

    for diff in differences:
        if diff>threshold:
            return 0

    return 1

**Ex 3.4** Test your function experimenting with different thresholds.
Select the best threshold on the whole data and report the accuracy in the Moodle.

You should achieve an accuracy higher than 0.70.

In [30]:
best_threshold=0
best_accuracy=0

for threshold in range(5,15):
    correct=0
    for i, row in df.iterrows():
        sentence, label=row[1], row[2]
        pred=detect_typos(sentence, threshold=threshold)
        if label==pred:
            correct+=1

    accuracy=correct/len(df)
    print(f"Threshold: {threshold}, Accuracy: {accuracy:.4f}")

    if accuracy>best_accuracy:
        best_accuracy=accuracy
        best_threshold=threshold

print(f"\nBest threshold: {best_threshold}")
print(f"Best accuracy: {best_accuracy:.4f}")


/tmp/ipykernel_230/245570523.py:7: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  sentence, label=row[1], row[2]


Threshold: 5, Accuracy: 0.5192
Threshold: 6, Accuracy: 0.5769
Threshold: 7, Accuracy: 0.6346
Threshold: 8, Accuracy: 0.6923
Threshold: 9, Accuracy: 0.7500
Threshold: 10, Accuracy: 0.7308
Threshold: 11, Accuracy: 0.7692
Threshold: 12, Accuracy: 0.7308
Threshold: 13, Accuracy: 0.7115
Threshold: 14, Accuracy: 0.6154

Best threshold: 11
Best accuracy: 0.7692


**Error Analysis**: in this exercise, we exploited the probability distribution of a Large Language Model to detect typos.
Did you notice any linguistic or grammatical feature that make detection more accurate? Do you think relying solely on threshold-based differences in log-probabilities is sufficient or should we use a more sophisticated and complete system? Write your comments in the box below.


### Moodle Submission

Extract the code to run exercise 3 from the notebook as a .py file (you might have to add some initial import instructions). Make sure the code runs and computes the accuracy correctly. Upload the code on the activity "Lab assignment 3" on the Moodle of the course. In the comment box on Moodle report your accuracy and your comments related to the error analys (box right above)